# Phase 4: Multi-Task Learning Training Pipeline

**Purpose**: Joint training of classification + NER with metadata integration

**Author**: GBC Biodata Inventory Team  
**Created**: 2025-10-31  
**Environment**: Google Colab with GPU (optimized for A100)

## Overview

This notebook implements Phase 4 multi-task learning with:
- **Shared RoBERTa encoder** for both classification and NER tasks
- **Metadata integration** (28 features) via projection and fusion layers
- **Fixed loss weighting**: λ₁=0.3 (classification), λ₂=0.7 (NER), λ₃=0.1 (auxiliary)
- **A100 optimizations**: Mixed precision training, batch size 32
- **Baseline comparison**: Evaluate against V2 single-task models

## Phase 4 Goals

- **Primary**: Maintain or improve NER F1 ≥ 0.749 (V2 baseline)
- **Secondary**: Maintain classification F1 ~ 0.898 (V2 baseline)
- **Hypothesis**: Multi-task learning + metadata improves NER via shared representations

## Features

- **TEST_MODE**: Quick validation with 50 samples and 3 epochs
- **Session Tracking**: Unique session IDs for reproducibility
- **Checkpointing**: Save best classification, best NER, best combined models
- **Gradient Monitoring**: Detect task conflicts and negative transfer
- **Google Drive Integration**: Persistent storage and archival

## Session Management

Each training session gets a unique ID: `YYYY-MM-DD-abcdef`
All outputs are saved to:
- Local: `experiments/{session_id}/`
- Drive: `MyDrive/inventory_2022/experiment_archives/{session_id}/`

---

## 🖥️ Colab Setup Requirements

**Before running this notebook:**

1. **GPU Runtime**: 
   - Go to: Runtime → Change runtime type → Hardware accelerator → **GPU**
   - **Recommended**: A100 GPU for fastest training (~1.5 hours)
   - **Alternative**: T4 or V100 GPU (~3-5 hours)

2. **RAM Setting**:
   - Select **High-RAM** if available (25.5 GB recommended)
   - Standard RAM (12.7 GB) should work but monitor usage

3. **Session Duration**:
   - **TEST_MODE=True**: 5-10 minutes
   - **TEST_MODE=False**: 1.5-5 hours (depending on GPU)
   - Keep tab open or enable browser notifications

4. **Important Notes**:
   - Colab may disconnect after 12 hours of inactivity
   - Training includes automatic checkpointing
   - All outputs saved to Google Drive for persistence

⚠️ **First-time users**: Run with `TEST_MODE = True` first to verify setup (takes ~5 min)

## Cell 1: Mount Google Drive and Setup Session

In [ ]:
# Mount Google Drive
from google.colab import drive
import os
import datetime
import random
import string

# Mount drive
drive.mount('/content/drive', force_remount=True)

# Set base paths
PROJECT_NAME = "inventory_2022"
DRIVE_BASE = f"/content/drive/MyDrive/{PROJECT_NAME}"

# Change to project directory
os.chdir(DRIVE_BASE)

print(f"✅ Mounted Google Drive")
print(f"📁 Working directory: {os.getcwd()}")

# Generate unique session ID: YYYY-MM-DD-abcdef
date_str = datetime.datetime.now().strftime("%Y-%m-%d")
random_suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=6))
SESSION_ID = f"{date_str}-{random_suffix}"

print(f"\n🔬 Session ID: {SESSION_ID}")

# Create experiment directories
EXPERIMENT_DIR = f"experiments/{SESSION_ID}"
ARCHIVE_DIR = f"{DRIVE_BASE}/experiment_archives/{SESSION_ID}"

os.makedirs(EXPERIMENT_DIR, exist_ok=True)
os.makedirs(ARCHIVE_DIR, exist_ok=True)

print(f"📂 Experiment directory: {EXPERIMENT_DIR}")
print(f"📦 Archive directory: {ARCHIVE_DIR}")

## Cell 2: Configuration - Phase 4 Multi-Task Learning

In [ ]:
# ========================================
# PHASE 4 CONFIGURATION - MULTI-TASK LEARNING
# ========================================

# TEST MODE: Set to True for quick validation (50 samples, 3 epochs)
TEST_MODE = False  # Set to True for testing, False for full training

# ========================================
# TRAINING CONFIGURATION
# ========================================
# Based on successful TEST_MODE validation run
# Fixed hyperparameters for Phase 4 MVP
# ========================================

CONFIG = {
    # Model configuration
    'model_name_or_path': 'roberta-base',
    'n_metadata_features': 28,  # CRITICAL: Must be 28 (usable features)
    'num_classes': 2,  # Binary classification
    'num_ner_labels': 3,  # BIO tagging: O, B-RESOURCE, I-RESOURCE
    'n_boolean_features': 10,  # For auxiliary task
    'n_numerical_features': 2,  # For auxiliary task
    
    # Training hyperparameters
    'learning_rate': 2e-5,  # Aligned with V2 baseline
    'weight_decay': 0.01,
    'warmup_steps': 500,
    'batch_size': 32,  # A100 optimization (use 16 for T4/V100)
    'epochs': 3 if TEST_MODE else 30,
    'gradient_clipping': 1.0,
    
    # Loss weighting (fixed for Phase 4 MVP)
    'lambda_classification': 0.3,  # Classification task weight
    'lambda_ner': 0.7,  # NER task weight (higher priority)
    'lambda_auxiliary': 0.1,  # Auxiliary regularization weight
    
    # Task-specific dropout
    'classification_dropout': 0.3,
    'ner_dropout': 0.1,
    
    # Early stopping
    'patience': 10,
    'min_delta': 0.001,
    
    # Data configuration
    'max_length_classif': 256,
    'max_length_ner': 512,
    'oversample_ner': True,  # Balance NER/classification samples
    
    # Mode flags
    'test_mode': TEST_MODE,
    'use_mixed_precision': True,  # A100 optimization
    'session_id': SESSION_ID
}

print("="*80)
print("PHASE 4: MULTI-TASK LEARNING CONFIGURATION")
print("="*80)
print(f"\nMode: {'TEST (50 samples, 3 epochs)' if TEST_MODE else 'FULL TRAINING (1,634 samples, 30 epochs)'}")
print(f"\nModel: {CONFIG['model_name_or_path']}")
print(f"Metadata features: {CONFIG['n_metadata_features']}")
print(f"\nTraining:")
print(f"  - Learning rate: {CONFIG['learning_rate']}")
print(f"  - Batch size: {CONFIG['batch_size']}")
print(f"  - Epochs: {CONFIG['epochs']}")
print(f"  - Warmup steps: {CONFIG['warmup_steps']}")
print(f"  - Gradient clipping: {CONFIG['gradient_clipping']}")
print(f"\nLoss Weighting:")
print(f"  - Classification (λ₁): {CONFIG['lambda_classification']}")
print(f"  - NER (λ₂): {CONFIG['lambda_ner']}")
print(f"  - Auxiliary (λ₃): {CONFIG['lambda_auxiliary']}")
print(f"\nDropout:")
print(f"  - Classification: {CONFIG['classification_dropout']}")
print(f"  - NER: {CONFIG['ner_dropout']}")
print(f"\nOptimizations:")
print(f"  - Mixed precision: {CONFIG['use_mixed_precision']} (A100)")
print(f"  - NER oversampling: {CONFIG['oversample_ner']}")

# V2 Baseline for comparison
V2_BASELINE = {
    'classification_f1': 0.898,
    'ner_f1': 0.749
}

print(f"\n{'='*80}")
print("V2 BASELINE TARGETS")
print(f"{'='*80}")
print(f"Classification F1: {V2_BASELINE['classification_f1']:.3f} (maintain)")
print(f"NER F1:           {V2_BASELINE['ner_f1']:.3f} (target ≥ this)")
print(f"\nPhase 4 Success Criteria:")
print(f"  ✓ Classification F1 ≥ 0.890 (within 1% of V2)")
print(f"  ✓ NER F1 ≥ 0.749 (match or exceed V2)")
print(f"  ✓ Multi-task learning enabled (shared encoder)")
print(f"  ✓ Metadata integration (28 features)")

# Estimate training time
if TEST_MODE:
    estimated_time = "5-10 minutes"
else:
    estimated_time = "2-3 hours (A100) or 4-6 hours (T4/V100)"

print(f"\nEstimated training time: {estimated_time}")
print(f"{'='*80}")

## Cell 3: Environment Setup and GPU Optimization

In [ ]:
import sys
import subprocess
import torch

# Add src to path
if '/content/drive/MyDrive/inventory_2022/src' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/inventory_2022/src')

print("📦 Installing/upgrading dependencies...")
# Install required packages (pin transformers for compatibility)
subprocess.run([
    'pip', 'install', '-q',
    'transformers==4.35.2',
    'datasets',
    'evaluate',
    'seqeval',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'pandas',
    'tqdm'
], check=False)

print("\n🔧 Importing modules...")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import logging
from datetime import datetime
from tqdm import tqdm

# Import Phase 4 modules with error handling
try:
    from src.models.multitask_model import BiomedicalMultiTaskModel, create_model
    from src.data.multitask_dataloader import create_multitask_dataloaders
    from src.train_multitask import MultiTaskTrainer
    from src.evaluate_multitask import MultiTaskEvaluator, generate_evaluation_report
    print("✅ Phase 4 modules imported successfully")
except ImportError as e:
    print(f"❌ Failed to import Phase 4 modules: {e}")
    print("\n🔍 Checking available modules:")
    import os
    src_path = '/content/drive/MyDrive/inventory_2022/src'
    if os.path.exists(src_path):
        print(f"   Available files in {src_path}:")
        for item in os.listdir(src_path):
            print(f"     - {item}")
    else:
        print(f"   ❌ src directory not found at {src_path}")
    raise

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# GPU Setup
print("\n" + "="*60)
print("GPU CONFIGURATION")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f} GB")
    
    # Configure mixed precision based on GPU type
    is_a100 = 'A100' in gpu_name
    if is_a100:
        print(f"\n   🚀 A100 GPU DETECTED - Optimizations enabled:")
        print(f"      ✅ Mixed precision training (FP16)")
        print(f"      ✅ Batch size: {CONFIG['batch_size']}")
        print(f"      ✅ Expected training time: ~1-1.5 hours (30 epochs)")
        CONFIG['use_mixed_precision'] = True
    else:
        print(f"\n   💡 Non-A100 GPU detected: {gpu_name}")
        # Conservative settings for T4/V100
        if 'T4' in gpu_name or 'V100' in gpu_name:
            CONFIG['use_mixed_precision'] = True  # Usually safe
            print(f"      ✅ Mixed precision: enabled")
            print(f"      ⚠️  Monitor for NaN losses (uncommon but possible)")
            if CONFIG['batch_size'] > 16:
                print(f"      ⚠️  Batch size {CONFIG['batch_size']} may cause OOM")
                print(f"         Consider reducing to 16 if training fails")
            print(f"      ⏱  Expected training time: ~3-5 hours (30 epochs)")
        else:
            CONFIG['use_mixed_precision'] = False  # Safer for unknown GPUs
            print(f"      ⚠️  Mixed precision: DISABLED (unknown GPU)")
            print(f"      💡 Training will be slower but more stable")
            print(f"      ⏱  Expected training time: ~6-8 hours (30 epochs)")
    
    # Clear GPU memory
    torch.cuda.empty_cache()
    print(f"\n   Memory after cleanup:")
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    print(f"      Allocated: {allocated:.2f} GB")
    print(f"      Reserved:  {reserved:.2f} GB")
    print(f"      Free:      {gpu_memory - reserved:.2f} GB")
    
    device = torch.device('cuda')
else:
    print("⚠️ No GPU available - training will be slow")
    print("   Consider using a GPU runtime for Phase 4 training")
    device = torch.device('cpu')
    CONFIG['use_mixed_precision'] = False  # Disable on CPU

CONFIG['device'] = device
print(f"\n✅ Environment setup complete")
print(f"   Device: {device}")

## Cell 4: Data Loading and 80/20 Split Creation

In [ ]:
print("="*80)
print("DATA LOADING AND PREPARATION")
print("="*80)

# Data paths
CLASSIF_TRAIN_PATH = "data/augmented/classif_train_with_metadata.csv"
NER_TRAIN_PATH = "data/augmented/ner_train_with_metadata.csv"

print(f"\n📂 Data Sources:")
print(f"   Classification: {CLASSIF_TRAIN_PATH}")
print(f"   NER: {NER_TRAIN_PATH}")

# Verify files exist with helpful instructions
if not Path(CLASSIF_TRAIN_PATH).exists():
    raise FileNotFoundError(
        f"\n❌ Classification data not found: {CLASSIF_TRAIN_PATH}\n\n"
        f"   📋 Required Action: Run data augmentation first\n"
        f"   📝 Command: python src/data_augmentation/augment_with_metadata.py\n\n"
        f"   OR ensure data/augmented/ directory exists with metadata files\n"
        f"   (augmented data is not tracked in git)"
    )
if not Path(NER_TRAIN_PATH).exists():
    raise FileNotFoundError(
        f"\n❌ NER data not found: {NER_TRAIN_PATH}\n\n"
        f"   📋 Required Action: Run data augmentation first\n"
        f"   📝 Command: python src/data_augmentation/augment_with_metadata.py\n\n"
        f"   OR ensure data/augmented/ directory exists with metadata files\n"
        f"   (augmented data is not tracked in git)"
    )

print("\n✅ Data files verified")

# Load full datasets for splitting
print("\n📥 Loading datasets...")
classif_df_full = pd.read_csv(CLASSIF_TRAIN_PATH)
ner_df_full = pd.read_csv(NER_TRAIN_PATH)

print(f"   Classification: {len(classif_df_full)} samples")
print(f"   NER: {len(ner_df_full)} samples")

# Create 80/20 train/val split
from sklearn.model_selection import train_test_split

print("\n✂️  Creating 80/20 train/val splits...")

# Classification split (stratified by curation_score)
classif_train, classif_val = train_test_split(
    classif_df_full,
    test_size=0.2,
    random_state=42,
    stratify=classif_df_full['curation_score']
)

# NER split (stratified by has_resource to maintain balance)
# Assuming has_resource column exists, otherwise use random split
if 'has_resource' in ner_df_full.columns:
    ner_train, ner_val = train_test_split(
        ner_df_full,
        test_size=0.2,
        random_state=42,
        stratify=ner_df_full['has_resource']
    )
else:
    ner_train, ner_val = train_test_split(
        ner_df_full,
        test_size=0.2,
        random_state=42
    )

print(f"\n📊 Split Statistics:")
print(f"\nClassification:")
print(f"   Train: {len(classif_train)} samples")
print(f"   Val:   {len(classif_val)} samples")
print(f"   Positive ratio (train): {classif_train['curation_score'].mean():.3f}")
print(f"   Positive ratio (val):   {classif_val['curation_score'].mean():.3f}")

print(f"\nNER:")
print(f"   Train: {len(ner_train)} samples")
print(f"   Val:   {len(ner_val)} samples")

# Save temporary splits for dataloader
split_dir = Path(EXPERIMENT_DIR) / "splits"
split_dir.mkdir(exist_ok=True)

classif_train_path = split_dir / "classif_train.csv"
classif_val_path = split_dir / "classif_val.csv"
ner_train_path = split_dir / "ner_train.csv"
ner_val_path = split_dir / "ner_val.csv"

classif_train.to_csv(classif_train_path, index=False)
classif_val.to_csv(classif_val_path, index=False)
ner_train.to_csv(ner_train_path, index=False)
ner_val.to_csv(ner_val_path, index=False)

print(f"\n💾 Splits saved to: {split_dir}")

# TEST_MODE: Use first 50 samples
if TEST_MODE:
    print(f"\n⚡ TEST_MODE: Using first 50 samples per task")
    classif_train = classif_train.head(50)
    classif_val = classif_val.head(25)
    ner_train = ner_train.head(50)
    ner_val = ner_val.head(25)
    
    # Re-save reduced splits
    classif_train.to_csv(classif_train_path, index=False)
    classif_val.to_csv(classif_val_path, index=False)
    ner_train.to_csv(ner_train_path, index=False)
    ner_val.to_csv(ner_val_path, index=False)
    
    print(f"   Classification: {len(classif_train)} train, {len(classif_val)} val")
    print(f"   NER: {len(ner_train)} train, {len(ner_val)} val")

print("\n✅ Data preparation complete")

## Cell 5: Create Multi-Task DataLoaders

In [ ]:
print("="*80)
print("CREATING MULTI-TASK DATALOADERS")
print("="*80)

# Create dataloaders
print(f"\n🔄 Loading tokenizer: {CONFIG['model_name_or_path']}")

train_loader, val_loader = create_multitask_dataloaders(
    classif_train_path=str(classif_train_path),
    ner_train_path=str(ner_train_path),
    classif_val_path=str(classif_val_path),
    ner_val_path=str(ner_val_path),
    tokenizer_name=CONFIG['model_name_or_path'],
    batch_size=CONFIG['batch_size'],
    test_mode=TEST_MODE,
    max_length_classif=CONFIG['max_length_classif'],
    max_length_ner=CONFIG['max_length_ner'],
    oversample_ner=CONFIG['oversample_ner'],
    num_workers=0  # Colab compatibility
)

print(f"\n✅ DataLoaders created:")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Batch size: {CONFIG['batch_size']}")

# Estimate steps per epoch
steps_per_epoch = len(train_loader)
total_steps = steps_per_epoch * CONFIG['epochs']

print(f"\n📊 Training Schedule:")
print(f"   Steps per epoch: {steps_per_epoch}")
print(f"   Total epochs: {CONFIG['epochs']}")
print(f"   Total steps: {total_steps}")
print(f"   Warmup steps: {CONFIG['warmup_steps']}")
print(f"   Warmup ratio: {CONFIG['warmup_steps']/total_steps:.1%}")

# Test batch
print(f"\n🧪 Testing batch loading...")
test_batch = next(iter(train_loader))
print(f"   Batch task: {test_batch['task'][0]}")
print(f"   Input IDs shape: {test_batch['input_ids'].shape}")
print(f"   Attention mask shape: {test_batch['attention_mask'].shape}")
print(f"   Labels shape: {test_batch['labels'].shape}")
print(f"   Metadata shape: {test_batch['metadata'].shape}")
print(f"   Metadata features: {test_batch['metadata'].shape[1]}")

# Verify metadata feature count matches config
assert test_batch['metadata'].shape[1] == CONFIG['n_metadata_features'], \
    f"Metadata mismatch: expected {CONFIG['n_metadata_features']}, got {test_batch['metadata'].shape[1]}"

print(f"\n✅ Batch loading test passed")

## Cell 6: Initialize Multi-Task Model and Trainer

In [ ]:
print("="*80)
print("MODEL INITIALIZATION")
print("="*80)

# Create model
print(f"\n🏗️  Creating BiomedicalMultiTaskModel...")
model = create_model(CONFIG)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Model size: ~{total_params * 4 / 1024**2:.1f} MB (FP32)")

# Model architecture summary
print(f"\n🏛️  Architecture:")
print(f"   Encoder: {CONFIG['model_name_or_path']}")
print(f"   Hidden size: {model.hidden_size}")
print(f"   Metadata features: {model.n_metadata_features}")
print(f"   Classification classes: {model.num_classes}")
print(f"   NER labels: {model.num_ner_labels}")
print(f"\n   Components:")
print(f"     1. Shared RoBERTa encoder")
print(f"     2. Metadata projection ({CONFIG['n_metadata_features']} → {model.hidden_size})")
print(f"     3. Fusion layer (text + metadata)")
print(f"     4. Classification head (dropout={CONFIG['classification_dropout']})")
print(f"     5. NER head (dropout={CONFIG['ner_dropout']})")
print(f"     6. Auxiliary heads (metadata prediction)")

# Create output directory
output_dir = Path(EXPERIMENT_DIR) / "multitask_training"
output_dir.mkdir(exist_ok=True)

print(f"\n📁 Output directory: {output_dir}")

# Initialize trainer
print(f"\n🎯 Initializing MultiTaskTrainer...")
trainer = MultiTaskTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=CONFIG,
    device=device,
    output_dir=str(output_dir)
)

print(f"\n✅ Trainer initialized:")
print(f"   Optimizer: AdamW (lr={CONFIG['learning_rate']}, wd={CONFIG['weight_decay']})")
print(f"   Scheduler: Linear warmup + decay")
print(f"   Loss weights: λ_classif={CONFIG['lambda_classification']}, λ_ner={CONFIG['lambda_ner']}, λ_aux={CONFIG['lambda_auxiliary']}")
print(f"   Early stopping: patience={CONFIG['patience']}")
print(f"   Gradient clipping: {CONFIG['gradient_clipping']}")
print(f"   Mixed precision: {CONFIG['use_mixed_precision']}")

# Validate trainer interface (ensure all required methods exist)
print(f"\n🔍 Validating trainer interface...")
required_attrs = ['train', 'history', 'optimizer', 'scheduler']
required_methods = ['train']

missing = []
for attr in required_attrs:
    if not hasattr(trainer, attr):
        missing.append(f"attribute '{attr}'")

for method in required_methods:
    if not hasattr(trainer, method) or not callable(getattr(trainer, method)):
        missing.append(f"method '{method}'")

if missing:
    raise AttributeError(
        f"❌ Trainer missing required interfaces: {', '.join(missing)}\n"
        f"   Check that MultiTaskTrainer in src/train_multitask.py is complete"
    )

print(f"   ✅ All required trainer interfaces present")
print(f"   ✅ Ready to begin training")

# Save configuration
config_path = output_dir / "config.json"
with open(config_path, 'w') as f:
    # Convert device to string for JSON serialization
    config_to_save = CONFIG.copy()
    config_to_save['device'] = str(device)
    json.dump(config_to_save, f, indent=2)

print(f"\n💾 Configuration saved to: {config_path}")
print(f"\n✅ Ready to train!")

## Cell 7: Training Loop with Progress Tracking

In [ ]:
print("="*80)
print(f"STARTING PHASE 4 TRAINING: {SESSION_ID}")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {'TEST' if TEST_MODE else 'FULL TRAINING'}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Device: {device}")
print("="*80)

# Start training
import time
start_time = time.time()

try:
    trainer.train(num_epochs=CONFIG['epochs'])
    
    training_time = time.time() - start_time
    
    print("\n" + "="*80)
    print("TRAINING COMPLETED SUCCESSFULLY")
    print("="*80)
    print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Total time: {training_time/60:.1f} minutes ({training_time/3600:.2f} hours)")
    print(f"\n📊 Best Results:")
    print(f"   Classification F1: {trainer.best_classif_f1:.4f}")
    print(f"   NER F1:           {trainer.best_ner_f1:.4f}")
    print(f"   Combined F1:      {trainer.best_combined_f1:.4f}")
    
    # Compare to V2 baseline
    print(f"\n📈 Comparison to V2 Baseline:")
    classif_delta = trainer.best_classif_f1 - V2_BASELINE['classification_f1']
    classif_pct = (classif_delta / V2_BASELINE['classification_f1']) * 100
    ner_delta = trainer.best_ner_f1 - V2_BASELINE['ner_f1']
    ner_pct = (ner_delta / V2_BASELINE['ner_f1']) * 100
    
    print(f"   Classification: {trainer.best_classif_f1:.4f} vs {V2_BASELINE['classification_f1']:.4f} ({classif_delta:+.4f}, {classif_pct:+.2f}%)")
    print(f"   NER:           {trainer.best_ner_f1:.4f} vs {V2_BASELINE['ner_f1']:.4f} ({ner_delta:+.4f}, {ner_pct:+.2f}%)")
    
    # Phase 4 success criteria
    print(f"\n🎯 Phase 4 Success Criteria:")
    classif_pass = trainer.best_classif_f1 >= 0.890
    ner_pass = trainer.best_ner_f1 >= 0.749
    print(f"   Classification F1 ≥ 0.890: {'✅ PASS' if classif_pass else '❌ FAIL'} ({trainer.best_classif_f1:.4f})")
    print(f"   NER F1 ≥ 0.749:           {'✅ PASS' if ner_pass else '❌ FAIL'} ({trainer.best_ner_f1:.4f})")
    print(f"   Multi-task learning:       ✅ ENABLED")
    print(f"   Metadata integration:      ✅ ENABLED (28 features)")
    
    overall_pass = classif_pass and ner_pass
    print(f"\n   Overall Phase 4 Status: {'✅ SUCCESS' if overall_pass else '⚠️  PARTIAL SUCCESS'}")
    
except torch.cuda.OutOfMemoryError:
    print("\n" + "="*60)
    print("❌ GPU OUT OF MEMORY!")
    print("="*60)
    print("Current configuration:")
    print(f"   Batch size: {CONFIG['batch_size']}")
    print(f"   Max length (classification): {CONFIG['max_length_classif']}")
    print(f"   Max length (NER): {CONFIG['max_length_ner']}")
    print("\n💡 Suggestions to fix:")
    print("   1. Reduce batch_size from 32 to 16 (edit Cell 4)")
    print("   2. Reduce max_length_ner from 512 to 384 (edit Cell 4)")
    print("   3. Disable mixed_precision (set to False in Cell 6)")
    print("   4. Restart runtime: Runtime → Restart runtime")
    print("\n📊 Current GPU usage:")
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        print(f"   Allocated: {allocated:.2f} GB")
        print(f"   Reserved: {reserved:.2f} GB")
    raise

except RuntimeError as e:
    error_str = str(e)
    if "CUDA" in error_str or "cuda" in error_str:
        print("\n" + "="*60)
        print(f"❌ CUDA ERROR")
        print("="*60)
        print(f"Error message: {error_str}")
        print("\n💡 This may indicate:")
        print("   - GPU disconnection")
        print("   - Driver crash")
        print("   - Memory corruption")
        print("   - Incompatible CUDA operations")
        print("\n🔧 Try these fixes:")
        print("   1. Runtime → Restart runtime")
        print("   2. Disable mixed precision (Cell 6)")
        print("   3. Check GPU availability (Runtime → View resources)")
    raise
    
except KeyboardInterrupt:
    print("\n" + "="*60)
    print("⚠️  TRAINING INTERRUPTED BY USER")
    print("="*60)
    training_time = time.time() - start_time
    print(f"Time elapsed: {training_time/60:.1f} minutes")
    
    # Save interrupt checkpoint
    print(f"\n💾 Saving interrupt checkpoint...")
    interrupt_path = output_dir / "checkpoint_interrupt.pt"
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'epoch': len(trainer.history.get('train_loss', [])),
        'config': CONFIG,
        'training_time': training_time
    }, interrupt_path)
    print(f"✅ Checkpoint saved to: {interrupt_path}")
    print(f"\n📝 To resume training:")
    print(f"   1. Load checkpoint: torch.load('{interrupt_path}')")
    print(f"   2. Restore model: model.load_state_dict(checkpoint['model_state_dict'])")
    print(f"   3. Continue from epoch: {len(trainer.history.get('train_loss', []))}")
    
except Exception as e:
    print("\n" + "="*60)
    print("❌ TRAINING FAILED")
    print("="*60)
    print(f"Error: {str(e)}")
    print("\nFull traceback:")
    import traceback
    traceback.print_exc()
    raise

## Cell 8: Evaluation and Baseline Comparison

In [ ]:
print("="*80)
print("FINAL EVALUATION")
print("="*80)

# Load best combined checkpoint
best_checkpoint_path = output_dir / "checkpoint_best_combined.pt"

if best_checkpoint_path.exists():
    print(f"\n📥 Loading best checkpoint: {best_checkpoint_path}")
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"   Checkpoint epoch: {checkpoint['epoch']}")
    print(f"   Checkpoint metrics:")
    for key, value in checkpoint['metrics'].items():
        print(f"     {key}: {value:.4f}")
else:
    print(f"\n⚠️  Best checkpoint not found, using current model state")

# Evaluate on validation set
print(f"\n🔍 Evaluating on validation set...")
evaluator = MultiTaskEvaluator(model, device)
results = evaluator.evaluate_both_tasks(val_loader)

# Prepare baseline metrics for comparison
baseline_metrics = {
    'classification': {
        'f1': V2_BASELINE['classification_f1'],
        'precision': 0.913,  # Approximate from V2
        'recall': 0.884,
        'accuracy': 0.921
    },
    'ner': {
        'f1_macro': V2_BASELINE['ner_f1'],
        'precision_macro': 0.756,  # Approximate from V2
        'recall_macro': 0.743,
        'accuracy': 0.881
    }
}

# Generate evaluation report
print(f"\n📝 Generating evaluation report...")
report_path = output_dir / "evaluation_report.txt"
report = generate_evaluation_report(
    results,
    baseline_metrics=baseline_metrics,
    output_path=str(report_path)
)

print(report)

# Save results JSON
results_path = output_dir / "evaluation_results.json"
with open(results_path, 'w') as f:
    # Convert numpy arrays to lists for JSON serialization
    results_serializable = {}
    for task, metrics in results.items():
        if isinstance(metrics, dict):
            results_serializable[task] = {}
            for key, value in metrics.items():
                if isinstance(value, (list, np.ndarray)):
                    results_serializable[task][key] = (
                        value.tolist() if isinstance(value, np.ndarray) else value
                    )
                elif isinstance(value, dict):
                    results_serializable[task][key] = value
                else:
                    results_serializable[task][key] = float(value) if isinstance(value, (np.floating, float)) else value
        else:
            results_serializable[task] = metrics
    
    json.dump(results_serializable, f, indent=2)

print(f"\n💾 Results saved to:")
print(f"   Report: {report_path}")
print(f"   JSON: {results_path}")

print("\n✅ Evaluation complete")

## Cell 9: Visualization - Training Curves

In [ ]:
print("="*80)
print("TRAINING VISUALIZATION")
print("="*80)

# Load training history
history = trainer.history

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(f'Phase 4 Multi-Task Training - {SESSION_ID}', fontsize=16, fontweight='bold')

# Plot 1: Total Loss
ax = axes[0, 0]
epochs = range(1, len(history['train_loss']) + 1)
ax.plot(epochs, history['train_loss'], 'b-', linewidth=2, label='Total Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Total Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Task-Specific Losses
ax = axes[0, 1]
ax.plot(epochs, history['train_classif_loss'], 'g-', linewidth=2, label='Classification Loss')
ax.plot(epochs, history['train_ner_loss'], 'r-', linewidth=2, label='NER Loss')
ax.plot(epochs, history['train_aux_loss'], 'orange', linewidth=2, label='Auxiliary Loss', linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Task-Specific Losses')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Validation F1 Scores
ax = axes[1, 0]
if history['val_classif_f1']:
    ax.plot(epochs, history['val_classif_f1'], 'g-', linewidth=2, marker='o', label='Classification F1')
    ax.axhline(y=V2_BASELINE['classification_f1'], color='g', linestyle='--', alpha=0.5, label='V2 Classif Baseline')
if history['val_ner_f1']:
    ax.plot(epochs, history['val_ner_f1'], 'r-', linewidth=2, marker='s', label='NER F1')
    ax.axhline(y=V2_BASELINE['ner_f1'], color='r', linestyle='--', alpha=0.5, label='V2 NER Baseline')
ax.set_xlabel('Epoch')
ax.set_ylabel('F1 Score')
ax.set_title('Validation F1 Scores vs V2 Baseline')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0.6, 1.0])

# Plot 4: Summary Statistics
ax = axes[1, 1]
ax.axis('off')

summary_text = f"""
PHASE 4 RESULTS vs V2 BASELINE
{'='*35}

Classification F1: {trainer.best_classif_f1:.4f}
  V2 Baseline:     {V2_BASELINE['classification_f1']:.4f}
  Delta:           {trainer.best_classif_f1 - V2_BASELINE['classification_f1']:+.4f} ({(trainer.best_classif_f1 - V2_BASELINE['classification_f1'])/V2_BASELINE['classification_f1']*100:+.2f}%)

NER F1:           {trainer.best_ner_f1:.4f}
  V2 Baseline:     {V2_BASELINE['ner_f1']:.4f}
  Delta:           {trainer.best_ner_f1 - V2_BASELINE['ner_f1']:+.4f} ({(trainer.best_ner_f1 - V2_BASELINE['ner_f1'])/V2_BASELINE['ner_f1']*100:+.2f}%)

Combined F1:      {trainer.best_combined_f1:.4f}

{'='*35}
CONFIGURATION
{'='*35}

Loss Weighting:
  Classification (λ₁): {CONFIG['lambda_classification']}
  NER (λ₂):           {CONFIG['lambda_ner']}
  Auxiliary (λ₃):     {CONFIG['lambda_auxiliary']}

Training:
  Epochs:             {CONFIG['epochs']}
  Batch size:         {CONFIG['batch_size']}
  Learning rate:      {CONFIG['learning_rate']}
  Warmup steps:       {CONFIG['warmup_steps']}

Phase 4 Success:
  Multi-task:         ✅ Enabled
  Metadata (28):      ✅ Integrated
  NER F1 ≥ 0.749:     {'✅' if trainer.best_ner_f1 >= 0.749 else '❌'} {trainer.best_ner_f1:.4f}
"""

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
        fontsize=10, verticalalignment='top',
        fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()

# Save figure
curves_path = output_dir / "training_curves.png"
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
print(f"\n💾 Training curves saved to: {curves_path}")

plt.show()

print("\n✅ Visualization complete")

## Cell 10: Archive Session to Google Drive

In [ ]:
import shutil

print("="*80)
print("SESSION ARCHIVAL")
print("="*80)

print(f"\n📦 Archiving session to Google Drive...")
print(f"   Source: {output_dir}")
print(f"   Destination: {ARCHIVE_DIR}")

# Copy entire output directory to archive
archive_output = Path(ARCHIVE_DIR) / "multitask_training"
if archive_output.exists():
    shutil.rmtree(archive_output)

shutil.copytree(output_dir, archive_output)

print(f"\n✅ Archived training outputs")

# Copy splits for reproducibility
archive_splits = Path(ARCHIVE_DIR) / "splits"
if archive_splits.exists():
    shutil.rmtree(archive_splits)

shutil.copytree(split_dir, archive_splits)

print(f"✅ Archived data splits")

# Create comprehensive session summary
summary_path = Path(ARCHIVE_DIR) / "SESSION_SUMMARY.md"

summary_content = f"""
# Phase 4 Multi-Task Training Session

**Session ID**: {SESSION_ID}  
**Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Mode**: {'TEST_MODE' if TEST_MODE else 'FULL_TRAINING'}  
**Device**: {device}  

## Configuration

### Model
- Base model: {CONFIG['model_name_or_path']}
- Metadata features: {CONFIG['n_metadata_features']}
- Classification classes: {CONFIG['num_classes']}
- NER labels: {CONFIG['num_ner_labels']}

### Training
- Learning rate: {CONFIG['learning_rate']}
- Batch size: {CONFIG['batch_size']}
- Epochs: {CONFIG['epochs']}
- Warmup steps: {CONFIG['warmup_steps']}
- Gradient clipping: {CONFIG['gradient_clipping']}
- Mixed precision: {CONFIG['use_mixed_precision']}

### Loss Weighting
- Classification (λ₁): {CONFIG['lambda_classification']}
- NER (λ₂): {CONFIG['lambda_ner']}
- Auxiliary (λ₃): {CONFIG['lambda_auxiliary']}

### Dropout
- Classification: {CONFIG['classification_dropout']}
- NER: {CONFIG['ner_dropout']}

## Results

### Best Metrics
- **Classification F1**: {trainer.best_classif_f1:.4f}
- **NER F1**: {trainer.best_ner_f1:.4f}
- **Combined F1**: {trainer.best_combined_f1:.4f}

### Comparison to V2 Baseline

| Task | Phase 4 | V2 Baseline | Delta | Change |
|------|---------|-------------|-------|--------|
| Classification F1 | {trainer.best_classif_f1:.4f} | {V2_BASELINE['classification_f1']:.4f} | {trainer.best_classif_f1 - V2_BASELINE['classification_f1']:+.4f} | {(trainer.best_classif_f1 - V2_BASELINE['classification_f1'])/V2_BASELINE['classification_f1']*100:+.2f}% |
| NER F1 | {trainer.best_ner_f1:.4f} | {V2_BASELINE['ner_f1']:.4f} | {trainer.best_ner_f1 - V2_BASELINE['ner_f1']:+.4f} | {(trainer.best_ner_f1 - V2_BASELINE['ner_f1'])/V2_BASELINE['ner_f1']*100:+.2f}% |

### Phase 4 Success Criteria

- ✅ Multi-task learning enabled (shared encoder)
- ✅ Metadata integration (28 features)
- {'✅' if trainer.best_classif_f1 >= 0.890 else '❌'} Classification F1 ≥ 0.890: {trainer.best_classif_f1:.4f}
- {'✅' if trainer.best_ner_f1 >= 0.749 else '❌'} NER F1 ≥ 0.749: {trainer.best_ner_f1:.4f}

**Overall Status**: {'✅ SUCCESS' if (trainer.best_classif_f1 >= 0.890 and trainer.best_ner_f1 >= 0.749) else '⚠️ PARTIAL SUCCESS'}

## Data

### Training Set
- Classification: {len(classif_train)} samples
- NER: {len(ner_train)} samples

### Validation Set
- Classification: {len(classif_val)} samples
- NER: {len(ner_val)} samples

## Files

### Checkpoints
- `checkpoint_best_classification.pt` - Best classification F1 model
- `checkpoint_best_ner.pt` - Best NER F1 model
- `checkpoint_best_combined.pt` - Best combined F1 model
- `checkpoint_final.pt` - Final epoch model

### Outputs
- `training_history.json` - Per-epoch metrics
- `evaluation_results.json` - Final evaluation metrics
- `evaluation_report.txt` - Detailed evaluation report
- `training_curves.png` - Visualization
- `config.json` - Complete configuration

### Data Splits
- `splits/classif_train.csv` - Classification training data
- `splits/classif_val.csv` - Classification validation data
- `splits/ner_train.csv` - NER training data
- `splits/ner_val.csv` - NER validation data

## Next Steps

1. Review evaluation report for detailed per-task metrics
2. Compare training curves to identify potential improvements
3. If NER F1 ≥ 0.749: Proceed to Phase 5 (inference pipeline integration)
4. If NER F1 < 0.749: Investigate causes and iterate on hyperparameters
5. Document findings and update experiment log

## Notes

- Phase 4 implements multi-task learning with metadata integration
- Fixed loss weighting used for MVP (λ₁=0.3, λ₂=0.7, λ₃=0.1)
- Baseline comparison uses V2 single-task models as reference
- All checkpoints and results archived to Google Drive for reproducibility
"""

with open(summary_path, 'w') as f:
    f.write(summary_content)

print(f"\n✅ Session summary created: {summary_path}")

# Display archive contents
print(f"\n📂 Archive contents:")
for item in sorted(Path(ARCHIVE_DIR).rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / (1024*1024)
        rel_path = item.relative_to(ARCHIVE_DIR)
        if size_mb > 0.1:  # Only show files > 100KB
            print(f"   {rel_path} ({size_mb:.1f} MB)")

# Calculate total archive size
total_size = sum(f.stat().st_size for f in Path(ARCHIVE_DIR).rglob('*') if f.is_file())
total_size_mb = total_size / (1024*1024)

print(f"\n{'='*80}")
print("SESSION SUMMARY")
print(f"{'='*80}")
print(f"Session ID: {SESSION_ID}")
print(f"Mode: {'TEST' if TEST_MODE else 'FULL TRAINING'}")
print(f"Training Time: {training_time/60:.1f} minutes")
print(f"Archive Size: {total_size_mb:.1f} MB")
print(f"Archive Location: {ARCHIVE_DIR}")

print(f"\n📊 Final Results:")
print(f"   Classification F1: {trainer.best_classif_f1:.4f} (V2: {V2_BASELINE['classification_f1']:.4f})")
print(f"   NER F1:           {trainer.best_ner_f1:.4f} (V2: {V2_BASELINE['ner_f1']:.4f})")
print(f"   Combined F1:      {trainer.best_combined_f1:.4f}")

print(f"\n🎉 PHASE 4 TRAINING SESSION COMPLETE!")
print(f"\n📝 Next steps:")
print(f"   1. Review {summary_path}")
print(f"   2. Examine evaluation_report.txt for detailed metrics")
print(f"   3. Analyze training_curves.png for convergence patterns")
print(f"   4. {'Proceed to Phase 5' if trainer.best_ner_f1 >= 0.749 else 'Iterate on hyperparameters'}")
print(f"   5. Document findings in experiment log")